In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer, MissingIndicator
import sqlite3
import json

## 📚 Part A: Conceptual Foundation


### Task 1 — Short Notes

**What is Data Analysis?**  
Data analysis means looking at raw data and trying to find useful information from it. We clean the data, explore it, and then use it to answer questions or make predictions.  
For example — if a bank wants to know which customers might not repay loans, they can analyze past data and find patterns.

**How to Plan a Data Science Project?**  
1. Understand the problem first — what are we trying to predict?  
2. Collect the data from different sources  
3. Clean the data — handle missing values, outliers etc  
4. Do feature engineering — create new useful columns  
5. Build the model  
6. Evaluate results  
In our case: predict if a customer will **default on a loan** (yes/no)

**How to Frame a Machine Learning Problem?**  
Our problem is a **CLASSIFICATION** problem.  
- Input (X) = customer features like age, income, credit score etc  
- Output (Y) = `default_flag` (0 = no default, 1 = default)  
- Algorithm type = **Supervised Learning** because we have labeled data  
- Metric = F1 Score or ROC-AUC (because data is imbalanced)

In [11]:
# A tensor is basically a container for numbers
# Different dimensions = different types of tensors

# 0D Tensor = Scalar (just one number)
scalar = np.array(42)
print("0D Tensor (Scalar):", scalar, "| Shape:", scalar.shape)

# 1D Tensor = Vector (a list of numbers)
vector = np.array([10, 20, 30, 40, 50])
print("1D Tensor (Vector):", vector, "| Shape:", vector.shape)

# 2D Tensor = Matrix (rows and columns - like a table)
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])
print("2D Tensor (Matrix):\n", matrix, "\nShape:", matrix.shape)

# 3D Tensor = Cube of numbers (like RGB image with 3 channels)
tensor_3d = np.array([[[1, 2], [3, 4]],
                       [[5, 6], [7, 8]]])
print("3D Tensor:\n", tensor_3d, "\nShape:", tensor_3d.shape)

# useful numpy operations
a = np.array([100, 200, 300, 400, 500])
print("\nArray:", a)
print("Mean:", np.mean(a), "| Sum:", np.sum(a))
print("Reshape to (5,1):\n", a.reshape(5, 1))

0D Tensor (Scalar): 42 | Shape: ()
1D Tensor (Vector): [10 20 30 40 50] | Shape: (5,)
2D Tensor (Matrix):
 [[1 2 3]
 [4 5 6]
 [7 8 9]] 
Shape: (3, 3)
3D Tensor:
 [[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]] 
Shape: (2, 2, 2)

Array: [100 200 300 400 500]
Mean: 300.0 | Sum: 1500
Reshape to (5,1):
 [[100]
 [200]
 [300]
 [400]
 [500]]


## Part-B


In [3]:
# importing csv dataset

df_csv = pd.read_csv("transactions.csv")
print(df_csv.head())

  customer_id  loan_amount loan_purpose  transaction_count  spending_ratio
0   CUST00001     25700.13          Car                 26         19.0994
1   CUST00002     19264.58     Business                  2         27.1723
2   CUST00003     23983.44    Education                 28         41.9300
3   CUST00004     58439.10          Car                 48         44.8451
4   CUST00005     63903.19    Education                 10         46.2541


In [4]:
# importing json dataset

df_json = pd.read_json("customer_metadata.json")
print(df_json.head())

  customer_id   age  gender region education_level employment_type
0   CUST00001  59.0  Female  South        Graduate   Self-Employed
1   CUST00002  49.0  Female   West       Secondary   Self-Employed
2   CUST00003  35.0  Female   East        Graduate             NaN
3   CUST00004  63.0  Female   East        Graduate   Self-Employed
4   CUST00005  28.0  Female  South        Graduate             NaN


In [5]:
# importing sql dataset

conn = sqlite3.connect("loan_repayment.db")
df_sql = pd.read_sql_query("SELECT * FROM loan_repayment_history", conn)
print(df_sql.head())


  customer_id  annual_income  credit_score  repayment_history
0   CUST00001   38384.982865    565.520304                  3
1   CUST00002   54156.786444    580.911557                  2
2   CUST00003   88523.013804    621.473062                  1
3   CUST00004  139662.121852    620.076082                  1
4   CUST00005   58780.063998    533.745555                  2


In [6]:
# importing dataset from api in from of json

with open("economic_indicators_api.json", 'r') as f:
    api_data = json.load(f)
df_api = pd.json_normalize(api_data['records'])
print(f"API data Shape: {df_api.shape}")
df_api.head()

API data Shape: (1000, 3)


,customer_id,join_date,default_flag
0,CUST00001,2022-03-04,0
1,CUST00002,2016-04-01,0
2,CUST00003,2015-04-13,0
3,CUST00004,2018-01-31,0
4,CUST00005,2017-09-30,0


In [7]:
# merge all the datsets

df = df_csv.merge(df_json, on="customer_id").merge(df_sql, on="customer_id").merge(df_api, on="customer_id")
display(df.head())
print(df.shape)

,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history,join_date,default_flag
0,CUST00001,25700.13,Car,26,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3,2022-03-04,0
1,CUST00002,19264.58,Business,2,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2,2016-04-01,0
2,CUST00003,23983.44,Education,28,41.9300,35.0,Female,East,Graduate,NaN,88523.013804,621.473062,1,2015-04-13,0
3,CUST00004,58439.10,Car,48,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1,2018-01-31,0
4,CUST00005,63903.19,Education,10,46.2541,28.0,Female,South,Graduate,NaN,58780.063998,533.745555,2,2017-09-30,0


(1000, 15)


## Part-C

In [8]:
# exploring the dataset

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1000 non-null   str    
 1   loan_amount        1000 non-null   float64
 2   loan_purpose       1000 non-null   str    
 3   transaction_count  1000 non-null   int64  
 4   spending_ratio     1000 non-null   float64
 5   age                950 non-null    float64
 6   gender             960 non-null    str    
 7   region             1000 non-null   str    
 8   education_level    1000 non-null   str    
 9   employment_type    940 non-null    str    
 10  annual_income      950 non-null    float64
 11  credit_score       960 non-null    float64
 12  repayment_history  1000 non-null   int64  
 13  join_date          1000 non-null   str    
 14  default_flag       1000 non-null   int64  
dtypes: float64(5), int64(3), str(7)
memory usage: 117.3 KB
None


In [9]:
print(df.describe())

         loan_amount  transaction_count  spending_ratio         age  \
count    1000.000000        1000.000000     1000.000000  950.000000   
mean    55609.235800          25.288000       28.928296   42.531579   
std     72016.026543          14.044868       15.871188   12.590350   
min      2375.180000           1.000000        1.238200   21.000000   
25%     20980.077500          13.000000       16.714375   31.250000   
50%     35477.625000          25.500000       27.333150   43.000000   
75%     62765.617500          37.000000       39.624425   53.000000   
max    907596.960000          49.000000       83.727900   64.000000   

       annual_income  credit_score  repayment_history  default_flag  
count   9.500000e+02    960.000000        1000.000000   1000.000000  
mean    1.490647e+05    599.094264           1.569000      0.007000  
std     2.124312e+05     85.412915           1.238052      0.083414  
min     1.569859e+04    290.000000           0.000000      0.000000  
25%     6.

In [10]:
print(df.isnull().sum())

customer_id           0
loan_amount           0
loan_purpose          0
transaction_count     0
spending_ratio        0
age                  50
gender               40
region                0
education_level       0
employment_type      60
annual_income        50
credit_score         40
repayment_history     0
join_date             0
default_flag          0
dtype: int64


In [13]:
# from ydata_profiling import ProfileReport

# profile = ProfileReport(
#     df,
#     title="Customer Credit Risk - Data Profiling Report",
#     explorative=True,   
#     minimal=False       
# )
# profile.to_file("data_profiling_report.html")

# print("Report saved! Open data_profiling_report.html in your browser.")

# profile.to_notebook_iframe()

In [18]:
# handling missing values with simple imputer

df_simple = df.copy()

mean_imputer = SimpleImputer(strategy="mean")
median_imputer = SimpleImputer(strategy="median")
frequent_imputer = SimpleImputer(strategy="most_frequent")

df_simple[["gender","employment_type"]] = frequent_imputer.fit_transform(df_simple[["gender","employment_type"]])
df_simple["age"] = mean_imputer.fit_transform(df_simple[["age"]])
df_simple[["credit_score","annual_income"]] = median_imputer.fit_transform(df_simple[["credit_score","annual_income"]])

print("after applying simple imputer")

print(df_simple.isnull().sum())
display(df_simple.head())



after applying simple imputer
customer_id          0
loan_amount          0
loan_purpose         0
transaction_count    0
spending_ratio       0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
credit_score         0
repayment_history    0
join_date            0
default_flag         0
dtype: int64


,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history,join_date,default_flag
0,CUST00001,25700.13,Car,26,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3,2022-03-04,0
1,CUST00002,19264.58,Business,2,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2,2016-04-01,0
2,CUST00003,23983.44,Education,28,41.9300,35.0,Female,East,Graduate,Salaried,88523.013804,621.473062,1,2015-04-13,0
3,CUST00004,58439.10,Car,48,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1,2018-01-31,0
4,CUST00005,63903.19,Education,10,46.2541,28.0,Female,South,Graduate,Salaried,58780.063998,533.745555,2,2017-09-30,0


In [20]:
# missing indicator and random sample imputation
df_random = df.copy()

def random_sample_imputer(df, col, random_state=18):
    null_count = df[col].isnull().sum()

    if null_count > 0:
        random_value = df[col].dropna().sample(
            n=null_count,
            replace=True,
            random_state=random_state
        ).values

        df_ = df.copy()
        df_.loc[df_[col].isnull(), col] = random_value

        return df_
    return df


miss_ind = MissingIndicator(features='missing-only')
miss_ind_array = miss_ind.fit_transform(df)

miss_ind_col = [df.columns[col] + '_missing' for col in miss_ind.features_]
df_miss_ind = pd.DataFrame(miss_ind_array.astype(int), columns=miss_ind_col)

print("\nNew indicator columns created:", miss_ind_col)
display(df_miss_ind.head())

df_random = pd.concat([df_random.reset_index(drop=True), df_miss_ind], axis=1)

print("\nShape after adding indicator columns:", df_random.shape)

for col in df_random.columns:
    df_random = random_sample_imputer(df_random,col)
    
print("after applying random impuataion")    
display(df_random.head())



New indicator columns created: ['age_missing', 'gender_missing', 'employment_type_missing', 'annual_income_missing', 'credit_score_missing']


,age_missing,gender_missing,employment_type_missing,annual_income_missing,credit_score_missing
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,1,0,0
3,0,0,0,0,0
4,0,0,1,0,0



Shape after adding indicator columns: (1000, 20)
after applying random impuataion


,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history,join_date,default_flag,age_missing,gender_missing,employment_type_missing,annual_income_missing,credit_score_missing
0,CUST00001,25700.13,Car,26,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3,2022-03-04,0,0,0,0,0,0
1,CUST00002,19264.58,Business,2,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2,2016-04-01,0,0,0,0,0,0
2,CUST00003,23983.44,Education,28,41.9300,35.0,Female,East,Graduate,Salaried,88523.013804,621.473062,1,2015-04-13,0,0,0,1,0,0
3,CUST00004,58439.10,Car,48,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1,2018-01-31,0,0,0,0,0,0
4,CUST00005,63903.19,Education,10,46.2541,28.0,Female,South,Graduate,Salaried,58780.063998,533.745555,2,2017-09-30,0,0,0,1,0,0


In [21]:
# knn imputer

df_knn = df.copy()

df_knn.drop(['customer_id', 'default_flag', 'join_date'], axis=1, inplace=True)

cols = ['gender', 'region', 'employment_type', 'loan_purpose', 'education_level']
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
df_knn[cols] = oe.fit_transform(df_knn[cols])

knn = KNNImputer(n_neighbors=10, weights='distance')
df_knn = pd.DataFrame(knn.fit_transform(df_knn), columns=df_knn.columns)

df_knn[cols] = df_knn[cols].round().astype(int)
df_knn[cols] = oe.inverse_transform(df_knn[cols])

print('After applying KNNImputer:')
display(df_knn.head(10))

After applying KNNImputer:


,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,25700.13,Car,26.0,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,19264.58,Business,2.0,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,23983.44,Education,28.0,41.9300,35.0,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,58439.10,Car,48.0,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,63903.19,Education,10.0,46.2541,28.0,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
5,13086.02,Education,39.0,17.2556,41.0,Female,East,Secondary,Salaried,458418.079889,654.852511,1.0
6,17727.11,Home,37.0,17.6691,59.0,Female,North,Primary,Salaried,47573.714010,548.864517,0.0
7,20571.90,Car,39.0,31.2673,39.0,Male,East,Graduate,Unemployed,177943.212904,587.497893,0.0
8,34501.93,Car,21.0,42.1541,43.0,Male,South,Graduate,Self-Employed,276619.631043,654.941926,2.0
9,9447.24,Car,28.0,27.3012,31.0,Female,South,Secondary,Self-Employed,259255.545054,548.654288,0.0


In [22]:
# iterative imputer or mice

df_mice = df.copy()

df_mice_dropped = df_mice.drop(['customer_id','default_flag','join_date'], axis=1)
df_mice.drop(['customer_id','default_flag','join_date'], axis=1, inplace=True)

cols = ['gender','region','employment_type','loan_purpose','education_level']
oe = OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=np.nan)
df_mice[cols] = oe.fit_transform(df_mice[cols])

mice = IterativeImputer(max_iter=175, random_state=18)
df_mice = pd.DataFrame(mice.fit_transform(df_mice), columns=df_mice.columns)

df_mice[cols] = df_mice[cols].round().astype(int)
df_mice[cols] = oe.inverse_transform(df_mice[cols])

print('After applying MICE Algorithm: ')
display(df_mice.head(10))

After applying MICE Algorithm: 


,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,25700.13,Car,26.0,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,19264.58,Business,2.0,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,23983.44,Education,28.0,41.9300,35.0,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,58439.10,Car,48.0,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,63903.19,Education,10.0,46.2541,28.0,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
5,13086.02,Education,39.0,17.2556,41.0,Female,East,Secondary,Salaried,458418.079889,654.852511,1.0
6,17727.11,Home,37.0,17.6691,59.0,Female,North,Primary,Salaried,47573.714010,548.864517,0.0
7,20571.90,Car,39.0,31.2673,39.0,Male,East,Graduate,Unemployed,177943.212904,587.497893,0.0
8,34501.93,Car,21.0,42.1541,43.0,Male,South,Graduate,Self-Employed,276619.631043,654.941926,2.0
9,9447.24,Car,28.0,27.3012,31.0,Female,South,Secondary,Self-Employed,259255.545054,548.654288,0.0


In [23]:
# complete case analysis

df_cca = df.copy()

print("Shape BEFORE dropping missing rows:", df_cca.shape)
print("Total missing values:\n", df_cca.isnull().sum())

df_cca = df_cca.dropna()

print("\nShape AFTER dropping missing rows:", df_cca.shape)
print("Total rows dropped:", df.shape[0] - df_cca.shape[0])
print("Total missing values after CCA:\n", df_cca.isnull().sum())

Shape BEFORE dropping missing rows: (1000, 15)
Total missing values:
 customer_id           0
loan_amount           0
loan_purpose          0
transaction_count     0
spending_ratio        0
age                  50
gender               40
region                0
education_level       0
employment_type      60
annual_income        50
credit_score         40
repayment_history     0
join_date             0
default_flag          0
dtype: int64

Shape AFTER dropping missing rows: (773, 15)
Total rows dropped: 227
Total missing values after CCA:
 customer_id          0
loan_amount          0
loan_purpose         0
transaction_count    0
spending_ratio       0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
credit_score         0
repayment_history    0
join_date            0
default_flag         0
dtype: int64


##### This tells that it is better to use simple imputer, knn imputer, iterative imputer and random sample imputer than just dropping the rows.